In [5]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("./data/lista_global_vars.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("../data/lista_global_vars.csv")

df = pd.read_csv(DATA_PATH)

print("DATA PATH:", DATA_PATH.resolve())
print("Shape:", df.shape)
print("\nColumns:")
for i, c in enumerate(df.columns):
    print(i, c)

DATA PATH: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/data/lista_global_vars.csv
Shape: (4024, 29)

Columns:
0 PAÍS
1 ETNIA.BN
2 EDAD
3 FUGAS.BN
4 ABUSOSUBS1
5 ABUSOSUBS2
6 CONVIVEN.1
7 CONVIVEN.2
8 CONVIVEN.3
9 CONVIVEN.4
10 CONVIVEN.5
11 CONVIVEN.6
12 AUTOEFIC.MEAN
13 AUTOEFIC.VAR
14 IMPULS.MEAN
15 IMPULS.MEDIAN
16 IMPULS.VAR
17 APOYO.MEAN
18 APOYO.MEDIAN
19 APOYO.VAR
20 MORAL.MEAN
21 MORAL.VAR
22 PORNO.T
23 GENERO_BIN_0
24 GENERO_BIN_1
25 GENERO_BIN_2
26 ORIENTSEX.BN_1
27 ORIENTSEX.BN_2
28 ORIENTSEX.BN_3


In [7]:
# ============================================================
# FINAL MODELS — SUBGROUP PERFORMANCE AUDIT
# PCA TRAIN-ONLY FINAL VERSION
# Manual subgroup definitions for ICREA data
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

MIN_N = 20
OUTPUT_DIR = Path("./final_subgroup_audit_PCA_trainonly")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    Path("./data/lista_global_vars.csv"),
    Path("../data/lista_global_vars.csv"),
    Path("../../data/lista_global_vars.csv"),
]

DATA_PATH = None
for p in DATA_CANDIDATES:
    if p.exists():
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError("Could not find lista_global_vars.csv. Adjust DATA_PATH manually.")

PREDICTION_FILES = {
    "victimization": Path("./final_victim/DT_victim_PCA_trainonly_TEST/predictions_with_probs.csv"),
    "perpetration": Path("./final_perpetrator/content/perpetrator_v3/predictions_with_probs.csv"),
    "overlap": Path(
        "./final_overlap/overlap_final/"
        "overlap_LOGREG_SW_pos1p5_PCA_trainonly_FINAL_PCA095_thr0p5_seed42/"
        "outputs/predictions_with_probs.csv"
    ),
}

print("DATA PATH:", DATA_PATH.resolve())
for outcome, p in PREDICTION_FILES.items():
    print(outcome, p.exists(), p.resolve())


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def detect_col(df, candidates):
    cols_lower = {str(c).lower(): c for c in df.columns}

    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]

    for c in df.columns:
        cl = str(c).lower()
        if any(cand.lower() in cl for cand in candidates):
            return c

    return None


def clean_binary_vector(x, name="vector"):
    """
    Converts input to a clean 1D integer binary vector.
    Handles Series, DataFrame, object dtype, floats, strings and NaNs.
    """
    # If duplicate column selection returned a DataFrame, take first column
    if isinstance(x, pd.DataFrame):
        print(f"WARNING: {name} is DataFrame with columns {list(x.columns)}. Using first column.")
        x = x.iloc[:, 0]

    # Convert to Series then numeric
    s = pd.Series(np.asarray(x).reshape(-1))
    s = pd.to_numeric(s, errors="coerce")

    if s.isna().any():
        bad_n = int(s.isna().sum())
        print(f"WARNING: {name} contains {bad_n} NaN/non-numeric values. Dropping them jointly later may be needed.")

    return s


def has_two_classes(y):
    s = clean_binary_vector(y, "y_true_check")
    s = pd.to_numeric(s, errors="coerce").dropna().astype(int)
    s = (s > 0).astype(int)
    return s.nunique() >= 2

def binary_metrics(y_true, y_pred):
    y_true_s = clean_binary_vector(y_true, "y_true")
    y_pred_s = clean_binary_vector(y_pred, "y_pred")

    # Joint valid mask
    valid = y_true_s.notna() & y_pred_s.notna()
    y_true_clean = y_true_s.loc[valid].astype(int).to_numpy()
    y_pred_clean = y_pred_s.loc[valid].astype(int).to_numpy()

    # Force any positive value to 1, just in case predictions are probabilities or floats
    y_true_clean = (y_true_clean > 0).astype(int)
    y_pred_clean = (y_pred_clean > 0).astype(int)

    # Debug if weird
    if len(y_true_clean) == 0:
        raise ValueError("No valid y_true/y_pred rows after cleaning.")

    print_once = False
    if set(np.unique(y_true_clean)) - {0, 1}:
        print_once = True
    if set(np.unique(y_pred_clean)) - {0, 1}:
        print_once = True

    if print_once:
        print("DEBUG y_true unique:", np.unique(y_true_clean, return_counts=True))
        print("DEBUG y_pred unique:", np.unique(y_pred_clean, return_counts=True))

    tn, fp, fn, tp = confusion_matrix(
        y_true_clean,
        y_pred_clean,
        labels=[0, 1]
    ).ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    f1 = 2 * ppv * recall / (ppv + recall) if (ppv + recall) > 0 else np.nan

    return {
        "n": int(len(y_true_clean)),
        "positives": int(np.sum(y_true_clean == 1)),
        "negatives": int(np.sum(y_true_clean == 0)),
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "recall_sensitivity": recall,
        "specificity": specificity,
        "precision_ppv": ppv,
        "npv": npv,
        "balanced_accuracy": balanced_accuracy_score(y_true_clean, y_pred_clean),
        "f1_positive": f1,
        "accuracy": accuracy_score(y_true_clean, y_pred_clean),
        "fpr": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
        "fnr": fn / (fn + tp) if (fn + tp) > 0 else np.nan,
    }

def load_predictions(path, outcome):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    dfp = pd.read_csv(path)

    y_true_col = detect_col(dfp, [
        f"y_true_{outcome}",
        "y_true_victim",
        "y_true_perp",
        "y_true_perpetration",
        "y_true_overlap",
        "y_true_intersect",
        "y_true",
        "target"
    ])

    y_pred_col = detect_col(dfp, [
        f"y_pred_{outcome}",
        "y_pred_victim",
        "y_pred_perp",
        "y_pred_perpetration",
        "y_pred_overlap",
        "y_pred_intersect",
        "y_pred",
        "prediction"
    ])

    y_prob_col = detect_col(dfp, [
        f"y_prob_{outcome}",
        "y_prob_victim",
        "y_prob_perp",
        "y_prob_perpetration",
        "y_prob_overlap",
        "y_prob_intersect",
        "y_prob",
        "probability",
        "score"
    ])

    idx_col = detect_col(dfp, [
        "idx_original",
        "original_idx",
        "idx",
        "index"
    ])

    if y_true_col is None or y_pred_col is None or idx_col is None:
        raise ValueError(
            f"Could not detect required columns for {outcome}. "
            f"Columns found: {list(dfp.columns)}"
        )

    rename_map = {
        idx_col: "idx_original",
        y_true_col: "y_true",
        y_pred_col: "y_pred",
    }

    if y_prob_col is not None:
        rename_map[y_prob_col] = "y_prob"

    dfp = dfp.rename(columns=rename_map)

    keep_cols = ["idx_original", "y_true", "y_pred"]
    if "y_prob" in dfp.columns:
        keep_cols.append("y_prob")

    dfp = dfp[keep_cols].copy()

    dfp["idx_original"] = dfp["idx_original"].astype(int)
    dfp["y_true"] = dfp["y_true"].astype(int)
    dfp["y_pred"] = dfp["y_pred"].astype(int)

    return dfp


# ------------------------------------------------------------
# Load analytical data
# ------------------------------------------------------------

df = pd.read_csv(DATA_PATH)
df = df.copy()

# idx_original in prediction files corresponds to the dataframe index used in notebooks
df["idx_original"] = df.index.astype(int)

print("\nAnalytical/raw subgroup data shape:", df.shape)
print("\nAvailable columns:")
for i, c in enumerate(df.columns):
    print(i, c)


# ------------------------------------------------------------
# Quick value counts for subgroup columns
# ------------------------------------------------------------

cols_check = [
    "PAÍS",
    "ETNIA.BN",
    "EDAD",
    "GENERO_BIN_1",
    "GENERO_BIN_2",
    "ORIENTSEX.BN_1",
    "ORIENTSEX.BN_2",
    "ORIENTSEX.BN_3",
]

print("\n=== VALUE COUNTS FOR SUBGROUP COLUMNS ===")
for col in cols_check:
    if col in df.columns:
        print("\n", col)
        print(df[col].value_counts(dropna=False).head(30))
    else:
        print("\nMISSING:", col)


# ------------------------------------------------------------
# Manual subgroup definitions — ICREA
# ------------------------------------------------------------

COUNTRY_COL = "PAÍS"
ETHNIC_COL = "ETNIA.BN"
AGE_COL = "EDAD"
GENDER_COLS = ["GENERO_BIN_1", "GENERO_BIN_2"]
ORIENT_COLS = ["ORIENTSEX.BN_1", "ORIENTSEX.BN_2", "ORIENTSEX.BN_3"]

SUBGROUP_SPECS = []

# Age groups
if AGE_COL in df.columns:
    age_num = pd.to_numeric(df[AGE_COL], errors="coerce")

    SUBGROUP_SPECS.append({
        "subgroup_variable": "age_group",
        "level": "14-15",
        "mask": age_num.isin([14, 15])
    })

    SUBGROUP_SPECS.append({
        "subgroup_variable": "age_group",
        "level": "16-17",
        "mask": age_num.isin([16, 17])
    })

# Country raw categories
if COUNTRY_COL in df.columns:
    for level in sorted(df[COUNTRY_COL].dropna().unique()):
        SUBGROUP_SPECS.append({
            "subgroup_variable": "country_raw",
            "level": str(level),
            "mask": df[COUNTRY_COL] == level
        })

# Ethnic minority raw categories
if ETHNIC_COL in df.columns:
    for level in sorted(df[ETHNIC_COL].dropna().unique()):
        SUBGROUP_SPECS.append({
            "subgroup_variable": "ethnic_minority_raw",
            "level": str(level),
            "mask": df[ETHNIC_COL] == level
        })

# Gender dummy columns
for col in GENDER_COLS:
    if col in df.columns:
        SUBGROUP_SPECS.append({
            "subgroup_variable": "gender_dummy",
            "level": f"{col}=1",
            "mask": pd.to_numeric(df[col], errors="coerce") == 1
        })

# Sexual orientation dummy columns
for col in ORIENT_COLS:
    if col in df.columns:
        SUBGROUP_SPECS.append({
            "subgroup_variable": "sexual_orientation_dummy",
            "level": f"{col}=1",
            "mask": pd.to_numeric(df[col], errors="coerce") == 1
        })

print("\n=== DEFINED SUBGROUP SPECS ===")
for spec in SUBGROUP_SPECS:
    print(
        spec["subgroup_variable"],
        "|",
        spec["level"],
        "| n raw =",
        int(spec["mask"].sum())
    )


# ------------------------------------------------------------
# Run subgroup audit
# ------------------------------------------------------------

all_rows = []
rowlevel_outputs = {}

# Prebuild mask lookup by idx_original
mask_lookup = {}
for spec in SUBGROUP_SPECS:
    mask_series = pd.Series(
        spec["mask"].to_numpy(),
        index=df["idx_original"].to_numpy()
    )
    mask_lookup[(spec["subgroup_variable"], spec["level"])] = mask_series

for outcome, pred_path in PREDICTION_FILES.items():
    pred = load_predictions(pred_path, outcome)

    merged = pred.merge(df, on="idx_original", how="left")

    # Remove duplicated column names if any
    if merged.columns.duplicated().any():
        print("WARNING duplicated columns after merge:")
        print(merged.columns[merged.columns.duplicated()].tolist())
        merged = merged.loc[:, ~merged.columns.duplicated()].copy()

    print("\n======================================")
    print("Outcome:", outcome)
    print("Prediction file:", pred_path)
    print("Pred n:", len(pred))
    print("Merged n:", len(merged))
    print("Rows with missing PAÍS after merge:", merged["PAÍS"].isna().sum() if "PAÍS" in merged.columns else "PAÍS missing")

    rowlevel_outputs[outcome] = merged

    # Overall row
    overall_metrics = binary_metrics(
        merged["y_true"],
        merged["y_pred"]
    )

    all_rows.append({
        "outcome": outcome,
        "subgroup_variable": "overall",
        "level": "overall",
        **overall_metrics
    })

    # Subgroup rows
    for spec in SUBGROUP_SPECS:
        var = spec["subgroup_variable"]
        level = spec["level"]

        mask_series = mask_lookup[(var, level)]
        m = merged["idx_original"].map(mask_series).fillna(False).astype(bool)

        sub = merged.loc[m].copy()

        # Remove duplicated column names if any
        if sub.columns.duplicated().any():
            print("WARNING duplicated columns in subgroup:")
            print(sub.columns[sub.columns.duplicated()].tolist())
            sub = sub.loc[:, ~sub.columns.duplicated()].copy()

        # Skip too-small or one-class groups
        if len(sub) < MIN_N:
            continue

        if not has_two_classes(sub["y_true"]):
            continue

        metrics = binary_metrics(
            sub["y_true"],
            sub["y_pred"]
        )

        all_rows.append({
            "outcome": outcome,
            "subgroup_variable": var,
            "level": level,
            **metrics
        })


audit_df = pd.DataFrame(all_rows)

audit_df = audit_df.sort_values(
    by=["outcome", "subgroup_variable", "level"]
).reset_index(drop=True)

audit_path = OUTPUT_DIR / "subgroup_performance_audit_final_PCA_trainonly.csv"
audit_df.to_csv(audit_path, index=False)

# Save row-level merged files for traceability
for outcome, merged in rowlevel_outputs.items():
    merged.to_csv(
        OUTPUT_DIR / f"{outcome}_rowlevel_with_subgroups.csv",
        index=False
    )

# Excel output skipped because openpyxl may not be installed
print("Excel export skipped. CSV files were saved correctly.")

# ------------------------------------------------------------
# Pretty display
# ------------------------------------------------------------

display_cols = [
    "outcome", "subgroup_variable", "level", "n",
    "positives", "negatives",
    "TP", "FP", "TN", "FN",
    "recall_sensitivity", "specificity",
    "precision_ppv", "npv",
    "balanced_accuracy", "f1_positive", "accuracy",
    "fpr", "fnr"
]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2400)

print("\n=== SUBGROUP AUDIT FINAL PCA TRAIN-ONLY ===")
print(
    audit_df[display_cols]
    .to_string(index=False, float_format=lambda x: f"{x:.3f}")
)

print("\nSaved CSV to:")
print(audit_path.resolve())

print("\nSaved Excel to:")
print(excel_path.resolve())

DATA PATH: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/data/lista_global_vars.csv
victimization True /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_victim/DT_victim_PCA_trainonly_TEST/predictions_with_probs.csv
perpetration True /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_perpetrator/content/perpetrator_v3/predictions_with_probs.csv
overlap True /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_overlap/overlap_final/overlap_LOGREG_SW_pos1p5_PCA_trainonly_FINAL_PCA095_thr0p5_seed42/outputs/predictions_with_probs.csv

Analytical/raw subgroup data shape: (4024, 30)

Available columns:
0 PAÍS
1 ETNIA.BN
2 EDAD
3 FUGAS.BN
4 ABUSOSUBS1
5 ABUSOSUBS2
6 CONVIVEN.1
7 CONVIVEN.2
8 CONVIVEN.3
9 CONVIVEN.4
10 CONVIVEN.5
11 CONVIVEN.6
12 AUTOEFIC.MEAN
13 AUTOEFIC.VAR
14 IMPULS.MEAN
15 IMPULS.MEDIAN
16 IMPULS.VAR
17 APOYO.

In [8]:
from pathlib import Path
import pandas as pd

audit_path = Path("./final_subgroup_audit_PCA_trainonly/subgroup_performance_audit_final_PCA_trainonly.csv")

print("Exists:", audit_path.exists())
print("Path:", audit_path.resolve())

audit_df = pd.read_csv(audit_path)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2400)

print(audit_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

Exists: True
Path: /Users/legna/WORKSPACE/ICREA/app_overlap/results_repo/intersection_model/Version_Final/final_subgroup_audit_PCA_trainonly/subgroup_performance_audit_final_PCA_trainonly.csv
      outcome        subgroup_variable            level   n  positives  negatives  TP  FP  TN  FN  recall_sensitivity  specificity  precision_ppv   npv  balanced_accuracy  f1_positive  accuracy   fpr   fnr
      overlap                age_group            14-15 513         96        417  79 187 230  17               0.823        0.552          0.297 0.931              0.687        0.436     0.602 0.448 0.177
      overlap                age_group            16-17 429         82        347  65 168 179  17               0.793        0.516          0.279 0.913              0.654        0.413     0.569 0.484 0.207
      overlap              country_raw                1 848        158        690 129 315 375  29               0.816        0.543          0.291 0.928              0.680        0.429     0.